# Kinthread r7 on any account (no shared folder needed)
Reads the dataset, the RL reader and the real questions from the project's link-viewable Drive folder (gdown), trains r7 (real-wording rows at 30%, chosen and stopped by real dev, health checks) and writes checkpoints to this account's own Drive (`MyDrive/kinthread_r7`). Run all; run all again to resume.

In [ ]:
import os, torch, glob
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b v9-reader https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip -q install -U gdown
S = '/content/share'   # read-only copies from the project's link-viewable Drive folder
FOLDERS = {'dataset_v7': '1MNiQuramSX5ImytxEdvNEIN7SgFtT6k8', 'rl': '1eX0Xwl0cn_cSTmHbBHFEMV0ISqFMZ7wX', 'r6a': '1F8gdW-kwba5IHZyWsi__aD3mqnJqPONA'}
FILES = {'real_questions.tgz': '1yIanNsRCb6-BAkGIF_SIjp9ZMCRIotGy', 'star_r3.jsonl.gz': '1_qk0JL_lkzKBdv2vwHxWijI4yNrNPRtr',
         'star_r2.jsonl.gz': '1s_RQLHpNbWsttRTjaoFpV1NP5I4q3Veh', 'corrections.jsonl.gz': '1VW_dWGo2GkDdV9AKVWl-pxz-IFPrpEZp'}
os.makedirs(S, exist_ok=True)
for name, fid in FOLDERS.items():
    if not os.path.isdir(f'{S}/{name}'):
        !gdown -q --folder https://drive.google.com/drive/folders/{fid} -O {S}/{name}
for name, fid in FILES.items():
    if not os.path.exists(f'{S}/{name}'):
        !gdown -q https://drive.google.com/uc?id={fid} -O {S}/{name}
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {S}/real_questions.tgz -C data/real_questions 2>/dev/null
from google.colab import drive
drive.mount('/content/drive')   # outputs go to this account's own Drive
OUT = '/content/drive/MyDrive/kinthread_r7/runs/v10-r7'
os.makedirs(OUT, exist_ok=True)
INIT = next(p for p in (f'{S}/rl/best.pt', f'{S}/r6a/best.pt') if os.path.exists(p))
DATA = f'{S}/dataset_v7'
print('init', INIT, '| data', DATA, os.listdir(DATA), '| out', OUT)

In [ ]:
EXTRA = [f'{DATA}/reworded.jsonl.gz', 'data/pairs_v1.jsonl.gz', 'data/star_r1.jsonl.gz*3', f'{S}/star_r2.jsonl.gz*5', 'data/teach_rows.jsonl.gz*5',
         f'{S}/corrections.jsonl.gz', f'{S}/star_r3.jsonl.gz*4', 'data/teach_sg2.jsonl.gz*4']
EXTRA = ','.join(e for e in EXTRA if os.path.exists(e.split('*')[0]))
print('extra rows:', EXTRA)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_reader.py --arch v10 --run v10-r7 --out {OUT} --pool {DATA} --init {INIT} --steps 15000 --of_form --real_dev 300 --select_real --batch 64 --lr 3e-4 --warmup 500 --optim muon --amp --workers 2 --resume --eval_every 1500 --save_every 1000 --reworded {EXTRA} --reworded_dev data/reworded_dev.jsonl.gz --p_reworded 0.3 --ramp_steps 2000 --block 448

In [ ]:
!python -m eval.run_real --ckpt {OUT}/best.pt --split dev --adaptive 1.0 --out {OUT}/real_dev.json | tail -4